<a href="https://colab.research.google.com/github/peperjet/bc-ml/blob/main/dacon/dacon_260331.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [94]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import lightgbm as lgb

In [95]:
# 데이터 읽어오기
train = pd.read_csv('/content/train.csv')
test = pd.read_csv('/content/test.csv')
submission = pd.read_csv('/content/sample_submission.csv')

# 잘 불러와졌는지 상단 5줄만 확인!
train.head()

,id,type,fiberID,psfMag_u,psfMag_g,psfMag_r,psfMag_i,psfMag_z,fiberMag_u,fiberMag_g,...,petroMag_u,petroMag_g,petroMag_r,petroMag_i,petroMag_z,modelMag_u,modelMag_g,modelMag_r,modelMag_i,modelMag_z
0,0,QSO,601,23.198224,21.431953,21.314148,21.176553,21.171444,22.581309,21.644453,...,22.504317,21.431636,21.478312,21.145409,20.422446,22.749241,21.465534,21.364187,21.020605,21.147340
1,1,QSO,788,21.431355,20.708104,20.678850,20.703420,20.473229,21.868797,21.029773,...,21.360701,20.778968,20.889705,20.639812,20.646660,21.492955,20.758527,20.753925,20.693389,20.512314
2,2,QSO,427,17.851451,16.727898,16.679677,16.694640,16.641788,18.171890,17.033098,...,17.867253,16.738784,16.688874,16.744210,16.808006,17.818063,16.697434,16.641249,16.660177,16.688928
3,3,QSO,864,20.789900,20.040371,19.926909,19.843840,19.463270,21.039030,20.317165,...,20.433907,19.993727,19.985531,19.750917,19.455117,20.770711,20.001699,19.889798,19.758113,19.552855
4,4,STAR_RED_DWARF,612,26.454969,23.058767,21.471406,19.504961,18.389096,25.700632,23.629122,...,25.859229,22.426929,21.673551,19.610012,18.376141,24.877052,23.147993,21.475342,19.487330,18.375655


In [96]:
#데이터 보양 보기

train.head()
test.head()
train.shape, test.shape

((199991, 23), (10009, 22))

In [97]:
# 컬럼 이름 보기
print(train.columns)
print(test.columns)

Index(['id', 'type', 'fiberID', 'psfMag_u', 'psfMag_g', 'psfMag_r', 'psfMag_i',
       'psfMag_z', 'fiberMag_u', 'fiberMag_g', 'fiberMag_r', 'fiberMag_i',
       'fiberMag_z', 'petroMag_u', 'petroMag_g', 'petroMag_r', 'petroMag_i',
       'petroMag_z', 'modelMag_u', 'modelMag_g', 'modelMag_r', 'modelMag_i',
       'modelMag_z'],
      dtype='object')
Index(['id', 'fiberID', 'psfMag_u', 'psfMag_g', 'psfMag_r', 'psfMag_i',
       'psfMag_z', 'fiberMag_u', 'fiberMag_g', 'fiberMag_r', 'fiberMag_i',
       'fiberMag_z', 'petroMag_u', 'petroMag_g', 'petroMag_r', 'petroMag_i',
       'petroMag_z', 'modelMag_u', 'modelMag_g', 'modelMag_r', 'modelMag_i',
       'modelMag_z'],
      dtype='object')


In [98]:
# 데이터 타입 보기
train.info()
test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 199991 entries, 0 to 199990
Data columns (total 23 columns):
 #   Column      Non-Null Count   Dtype  
---  ------      --------------   -----  
 0   id          199991 non-null  int64  
 1   type        199991 non-null  object 
 2   fiberID     199991 non-null  int64  
 3   psfMag_u    199991 non-null  float64
 4   psfMag_g    199991 non-null  float64
 5   psfMag_r    199991 non-null  float64
 6   psfMag_i    199991 non-null  float64
 7   psfMag_z    199991 non-null  float64
 8   fiberMag_u  199991 non-null  float64
 9   fiberMag_g  199991 non-null  float64
 10  fiberMag_r  199991 non-null  float64
 11  fiberMag_i  199991 non-null  float64
 12  fiberMag_z  199991 non-null  float64
 13  petroMag_u  199991 non-null  float64
 14  petroMag_g  199991 non-null  float64
 15  petroMag_r  199991 non-null  float64
 16  petroMag_i  199991 non-null  float64
 17  petroMag_z  199991 non-null  float64
 18  modelMag_u  199991 non-null  float64
 19  mo

In [99]:
# 결측치 확인(빈칸 찾기)

train.isnull().sum()
test.isnull().sum()

,0
id,0
fiberID,0
psfMag_u,0
psfMag_g,0
psfMag_r,0
psfMag_i,0
psfMag_z,0
fiberMag_u,0
fiberMag_g,0
fiberMag_r,0


In [100]:
# 문자열 컬럼 확인
train.select_dtypes(include='object').columns
test.select_dtypes(include='object').columns

Index([], dtype='object')

In [101]:
# 기초 통계
train.describe()
test.describe()

,id,fiberID,psfMag_u,psfMag_g,psfMag_r,psfMag_i,psfMag_z,fiberMag_u,fiberMag_g,fiberMag_r,...,petroMag_u,petroMag_g,petroMag_r,petroMag_i,petroMag_z,modelMag_u,modelMag_g,modelMag_r,modelMag_i,modelMag_z
count,10009.000000,10009.000000,10009.000000,10009.000000,10009.000000,10009.000000,10009.000000,10009.000000,10009.000000,10009.000000,...,10009.000000,10009.000000,10009.000000,10009.000000,10009.000000,10009.000000,10009.000000,10009.000000,10009.000000,10009.000000
mean,204995.000000,359.327805,20.987400,19.878440,19.280218,18.873165,18.618385,21.184506,20.091376,19.497732,...,20.715429,19.462021,18.995001,18.616519,18.411996,20.739001,19.534844,18.935095,18.522308,18.281069
std,2889.493756,223.928862,2.111703,2.573890,1.709344,1.720713,1.702236,1.990940,1.865064,1.710070,...,2.807434,13.971203,1.979225,1.970463,2.373022,2.187025,1.957506,1.856617,1.796820,1.867582
min,199991.000000,1.000000,-7.248195,-42.663871,9.134712,-22.522266,13.349827,9.390439,8.188752,12.288183,...,-98.181975,-1348.068776,-23.908952,-8.356654,-64.917293,12.419765,13.617577,13.382832,12.955113,12.395695
25%,202493.000000,174.000000,19.655525,18.671025,18.037847,17.742016,17.424701,19.940430,18.892043,18.253627,...,19.249432,18.104265,17.475078,17.043615,16.805557,19.268471,18.064625,17.424393,16.971911,16.715685
50%,204995.000000,346.000000,20.854404,19.910333,19.444925,19.033283,18.594713,21.040735,20.071658,19.627904,...,20.371014,19.582955,19.197068,18.684269,18.171966,20.412702,19.541261,19.155839,18.635195,18.095883
75%,207497.000000,525.000000,22.160801,21.150040,20.489912,20.083814,19.878652,22.339213,21.402558,20.756306,...,21.803705,21.025719,20.428279,20.015701,19.815916,21.992969,20.981396,20.389215,19.970657,19.823902
max,209999.000000,1000.000000,37.681143,182.654452,31.883768,47.227391,34.946057,41.169991,47.160580,29.266687,...,65.392087,106.962571,41.850633,52.221528,74.747394,32.641240,28.814977,27.579664,26.471555,24.461973


In [102]:
# 정답 컬럼 숫자로 변환 및 X, y, X_test 분리
le = LabelEncoder()
train['type'] = le.fit_transform(train['type'])

y = train['type']
X = train.drop(['id', 'type'], axis=1)

X_test = test.drop(['id'], axis=1) # 'id' 컬럼은 더 이상 필요 없으므로 X_test에서 바로 제거

In [103]:
# 상단 데이터 보기
X.head()
X_test.head()

,fiberID,psfMag_u,psfMag_g,psfMag_r,psfMag_i,psfMag_z,fiberMag_u,fiberMag_g,fiberMag_r,fiberMag_i,...,petroMag_u,petroMag_g,petroMag_r,petroMag_i,petroMag_z,modelMag_u,modelMag_g,modelMag_r,modelMag_i,modelMag_z
0,251,23.817399,22.508963,20.981106,18.517316,17.076079,25.053890,23.167848,21.335901,18.835858,...,22.246697,22.796239,21.195315,18.584486,17.154284,25.391534,22.499435,21.011918,18.499341,17.091474
1,386,22.806983,21.937111,20.335770,20.000512,19.527369,22.498565,22.186000,20.618879,20.301204,...,21.729831,21.837511,20.196128,19.967204,19.683671,22.475338,21.853442,20.173169,19.796757,19.567372
2,232,21.024250,19.235669,18.304061,17.808608,17.380113,21.205546,19.439533,18.344433,17.909690,...,20.722629,18.710223,17.611851,17.158519,16.843986,20.579314,18.653338,17.562108,17.120529,16.708748
3,557,20.503424,20.286261,20.197204,20.162419,20.059832,20.976132,20.611498,20.567262,20.479318,...,20.329269,20.385262,20.129157,20.206574,20.212342,20.479879,20.280943,20.150499,20.206221,20.092909
4,75,24.244851,22.668237,21.239333,19.284777,18.235939,25.681860,22.935289,21.642456,19.624926,...,22.308298,22.957496,21.285033,19.299120,18.307526,25.489360,22.857290,21.191862,19.237964,18.280368


In [104]:
#크기확인

X.shape
X_test.shape

(10009, 21)

In [105]:
# 컬럼 확인

print(X.columns)
print(X_test.columns)

Index(['fiberID', 'psfMag_u', 'psfMag_g', 'psfMag_r', 'psfMag_i', 'psfMag_z',
       'fiberMag_u', 'fiberMag_g', 'fiberMag_r', 'fiberMag_i', 'fiberMag_z',
       'petroMag_u', 'petroMag_g', 'petroMag_r', 'petroMag_i', 'petroMag_z',
       'modelMag_u', 'modelMag_g', 'modelMag_r', 'modelMag_i', 'modelMag_z'],
      dtype='object')
Index(['fiberID', 'psfMag_u', 'psfMag_g', 'psfMag_r', 'psfMag_i', 'psfMag_z',
       'fiberMag_u', 'fiberMag_g', 'fiberMag_r', 'fiberMag_i', 'fiberMag_z',
       'petroMag_u', 'petroMag_g', 'petroMag_r', 'petroMag_i', 'petroMag_z',
       'modelMag_u', 'modelMag_g', 'modelMag_r', 'modelMag_i', 'modelMag_z'],
      dtype='object')


In [106]:
# 타입 확인

X.info()
X_test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 199991 entries, 0 to 199990
Data columns (total 21 columns):
 #   Column      Non-Null Count   Dtype  
---  ------      --------------   -----  
 0   fiberID     199991 non-null  int64  
 1   psfMag_u    199991 non-null  float64
 2   psfMag_g    199991 non-null  float64
 3   psfMag_r    199991 non-null  float64
 4   psfMag_i    199991 non-null  float64
 5   psfMag_z    199991 non-null  float64
 6   fiberMag_u  199991 non-null  float64
 7   fiberMag_g  199991 non-null  float64
 8   fiberMag_r  199991 non-null  float64
 9   fiberMag_i  199991 non-null  float64
 10  fiberMag_z  199991 non-null  float64
 11  petroMag_u  199991 non-null  float64
 12  petroMag_g  199991 non-null  float64
 13  petroMag_r  199991 non-null  float64
 14  petroMag_i  199991 non-null  float64
 15  petroMag_z  199991 non-null  float64
 16  modelMag_u  199991 non-null  float64
 17  modelMag_g  199991 non-null  float64
 18  modelMag_r  199991 non-null  float64
 19  mo

모델 만들기


In [107]:
model = RandomForestClassifier(
    n_estimators=50,
    max_depth=10,
    n_jobs=-1,
    random_state=42
)

In [108]:
# 학습

model.fit(X, y)

RandomForestClassifier(max_depth=10, n_estimators=50, n_jobs=-1,
                       random_state=42)

In [109]:
print(X.columns)
print(X_test.columns)

Index(['fiberID', 'psfMag_u', 'psfMag_g', 'psfMag_r', 'psfMag_i', 'psfMag_z',
       'fiberMag_u', 'fiberMag_g', 'fiberMag_r', 'fiberMag_i', 'fiberMag_z',
       'petroMag_u', 'petroMag_g', 'petroMag_r', 'petroMag_i', 'petroMag_z',
       'modelMag_u', 'modelMag_g', 'modelMag_r', 'modelMag_i', 'modelMag_z'],
      dtype='object')
Index(['fiberID', 'psfMag_u', 'psfMag_g', 'psfMag_r', 'psfMag_i', 'psfMag_z',
       'fiberMag_u', 'fiberMag_g', 'fiberMag_r', 'fiberMag_i', 'fiberMag_z',
       'petroMag_u', 'petroMag_g', 'petroMag_r', 'petroMag_i', 'petroMag_z',
       'modelMag_u', 'modelMag_g', 'modelMag_r', 'modelMag_i', 'modelMag_z'],
      dtype='object')


In [110]:
# 가벼운 모델로 재시도

model = RandomForestClassifier(
    n_estimators=20,
    max_depth=5,
    n_jobs=-1,
    random_state=42
)

model.fit(X, y)
pred = model.predict(X_test)

submission['type'] = le.inverse_transform(pred)
submission.to_csv('submission.csv', index=False)

In [111]:
# 파생변수 색지수

X['psf_u_g'] = X['psfMag_u'] - X['psfMag_g']
X['psf_g_r'] = X['psfMag_g'] - X['psfMag_r']
X['psf_r_i'] = X['psfMag_r'] - X['psfMag_i']
X['psf_i_z'] = X['psfMag_i'] - X['psfMag_z']

X_test['psf_u_g'] = X_test['psfMag_u'] - X_test['psfMag_g']
X_test['psf_g_r'] = X_test['psfMag_g'] - X_test['psfMag_r']
X_test['psf_r_i'] = X_test['psfMag_r'] - X_test['psfMag_i']
X_test['psf_i_z'] = X_test['psfMag_i'] - X_test['psfMag_z']


# 값 확인
X[['psfMag_u', 'psfMag_g', 'psf_u_g']].head()

,psfMag_u,psfMag_g,psf_u_g
0,23.198224,21.431953,1.766271
1,21.431355,20.708104,0.723251
2,17.851451,16.727898,1.123553
3,20.789900,20.040371,0.749529
4,26.454969,23.058767,3.396202


In [112]:
# 전체확인
X.head()

,fiberID,psfMag_u,psfMag_g,psfMag_r,psfMag_i,psfMag_z,fiberMag_u,fiberMag_g,fiberMag_r,fiberMag_i,...,petroMag_z,modelMag_u,modelMag_g,modelMag_r,modelMag_i,modelMag_z,psf_u_g,psf_g_r,psf_r_i,psf_i_z
0,601,23.198224,21.431953,21.314148,21.176553,21.171444,22.581309,21.644453,21.657571,21.387653,...,20.422446,22.749241,21.465534,21.364187,21.020605,21.147340,1.766271,0.117805,0.137595,0.005109
1,788,21.431355,20.708104,20.678850,20.703420,20.473229,21.868797,21.029773,20.967054,20.937731,...,20.646660,21.492955,20.758527,20.753925,20.693389,20.512314,0.723251,0.029254,-0.024570,0.230191
2,427,17.851451,16.727898,16.679677,16.694640,16.641788,18.171890,17.033098,16.999682,17.095999,...,16.808006,17.818063,16.697434,16.641249,16.660177,16.688928,1.123553,0.048221,-0.014963,0.052852
3,864,20.789900,20.040371,19.926909,19.843840,19.463270,21.039030,20.317165,20.217898,20.073852,...,19.455117,20.770711,20.001699,19.889798,19.758113,19.552855,0.749529,0.113462,0.083069,0.380569
4,612,26.454969,23.058767,21.471406,19.504961,18.389096,25.700632,23.629122,21.742750,19.861718,...,18.376141,24.877052,23.147993,21.475342,19.487330,18.375655,3.396202,1.587361,1.966445,1.115864


In [113]:
# 컬럼 목록 확인

print(X.columns)

Index(['fiberID', 'psfMag_u', 'psfMag_g', 'psfMag_r', 'psfMag_i', 'psfMag_z',
       'fiberMag_u', 'fiberMag_g', 'fiberMag_r', 'fiberMag_i', 'fiberMag_z',
       'petroMag_u', 'petroMag_g', 'petroMag_r', 'petroMag_i', 'petroMag_z',
       'modelMag_u', 'modelMag_g', 'modelMag_r', 'modelMag_i', 'modelMag_z',
       'psf_u_g', 'psf_g_r', 'psf_r_i', 'psf_i_z'],
      dtype='object')


In [114]:
# 통계 1개

cols = [c for c in X.columns if 'Mag' in c]

X['mag_mean'] = X[cols].mean(axis=1)
X['mag_std'] = X[cols].std(axis=1)

X_test['mag_mean'] = X_test[cols].mean(axis=1)
X_test['mag_std'] = X_test[cols].std(axis=1)

In [115]:
# 통계 값 확인

X[['mag_mean', 'mag_std']].head()

,mag_mean,mag_std
0,21.593258,0.666084
1,20.919446,0.363768
2,17.016278,0.493565
3,20.054956,0.433491
4,21.696652,2.647524


In [116]:
# 기존 값이랑 같이 보기

cols = [c for c in X.columns if 'Mag' in c][:5]
X[cols + ['mag_mean', 'mag_std']].head()

,psfMag_u,psfMag_g,psfMag_r,psfMag_i,psfMag_z,mag_mean,mag_std
0,23.198224,21.431953,21.314148,21.176553,21.171444,21.593258,0.666084
1,21.431355,20.708104,20.678850,20.703420,20.473229,20.919446,0.363768
2,17.851451,16.727898,16.679677,16.694640,16.641788,17.016278,0.493565
3,20.789900,20.040371,19.926909,19.843840,19.463270,20.054956,0.433491
4,26.454969,23.058767,21.471406,19.504961,18.389096,21.696652,2.647524


In [117]:
# 컬럼 생성 확인

print(X.columns)

Index(['fiberID', 'psfMag_u', 'psfMag_g', 'psfMag_r', 'psfMag_i', 'psfMag_z',
       'fiberMag_u', 'fiberMag_g', 'fiberMag_r', 'fiberMag_i', 'fiberMag_z',
       'petroMag_u', 'petroMag_g', 'petroMag_r', 'petroMag_i', 'petroMag_z',
       'modelMag_u', 'modelMag_g', 'modelMag_r', 'modelMag_i', 'modelMag_z',
       'psf_u_g', 'psf_g_r', 'psf_r_i', 'psf_i_z', 'mag_mean', 'mag_std'],
      dtype='object')


In [118]:
model2 = RandomForestClassifier(
    n_estimators=30,
    max_depth=8,
    n_jobs=-1,
    random_state=42
)

model2.fit(X, y)
pred2 = model2.predict(X_test)

submission['type'] = le.inverse_transform(pred2)
submission.to_csv('submission_fe.csv', index=False)

In [119]:
model_mid = RandomForestClassifier(
    n_estimators=20,
    max_depth=6,
    min_samples_leaf=3,
    max_features='sqrt',
    n_jobs=-1,
    random_state=42
)

model_mid.fit(X, y)
pred_mid = model_mid.predict(X_test)

submission['type'] = le.inverse_transform(pred_mid)
submission.to_csv('submission_mid.csv', index=False)

In [120]:
submission.head()

,id,STAR_WHITE_DWARF,STAR_CATY_VAR,STAR_BROWN_DWARF,SERENDIPITY_RED,REDDEN_STD,STAR_BHB,GALAXY,SERENDIPITY_DISTANT,QSO,...,STAR_RED_DWARF,ROSAT_D,STAR_PN,SERENDIPITY_FIRST,STAR_CARBON,SPECTROPHOTO_STD,STAR_SUB_DWARF,SERENDIPITY_MANUAL,SERENDIPITY_BLUE,type
0,199991,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,STAR_RED_DWARF
1,199992,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,QSO
2,199993,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,GALAXY
3,199994,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,SERENDIPITY_BLUE
4,199995,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,STAR_RED_DWARF


In [121]:
submission = pd.read_csv('/content/sample_submission.csv')

proba_mid = model_mid.predict_proba(X_test)

# Get class names from the LabelEncoder
class_names = le.classes_

# Create a DataFrame from the probabilities with correct class names as columns
proba_df = pd.DataFrame(proba_mid, columns=class_names)

# Assign the probabilities to the submission DataFrame, aligning by column names
# Assuming 'id' is the first column in submission.csv, we align from the second column onward.
submission.iloc[:, 1:] = proba_df

submission.to_csv('submission_mid.csv', index=False)

In [122]:
submission.head()

,id,STAR_WHITE_DWARF,STAR_CATY_VAR,STAR_BROWN_DWARF,SERENDIPITY_RED,REDDEN_STD,STAR_BHB,GALAXY,SERENDIPITY_DISTANT,QSO,SKY,STAR_RED_DWARF,ROSAT_D,STAR_PN,SERENDIPITY_FIRST,STAR_CARBON,SPECTROPHOTO_STD,STAR_SUB_DWARF,SERENDIPITY_MANUAL,SERENDIPITY_BLUE
0,199991,0.001783,0.006145,0.000013,0.006071,0.000256,0.000204,0.000139,0.002337,0.135623,0.000000,0.00000,0.000306,0.010733,0.000059,0.000257,0.000000,0.835922,0.000153,0.000000
1,199992,0.035121,0.430287,0.000002,0.174342,0.033099,0.014894,0.287350,0.000808,0.003287,0.001962,0.00000,0.004082,0.000422,0.006576,0.000104,0.000106,0.005805,0.000323,0.001430
2,199993,0.962330,0.009067,0.001846,0.006918,0.000088,0.000014,0.003803,0.000049,0.000009,0.000000,0.00017,0.000035,0.000015,0.010375,0.003399,0.000000,0.000654,0.000110,0.001116
3,199994,0.000372,0.170578,0.000000,0.032130,0.703540,0.039709,0.034347,0.000270,0.000012,0.000009,0.00000,0.004548,0.000000,0.000000,0.005341,0.000110,0.000018,0.000000,0.009018
4,199995,0.001403,0.003633,0.000013,0.006861,0.000010,0.000248,0.000339,0.000287,0.018411,0.000000,0.00000,0.000019,0.001402,0.000095,0.000562,0.000000,0.966527,0.000188,0.000000


In [123]:
submission.iloc[:, 1:].sum(axis=1).head()

,0
0,1.0
1,1.0
2,1.0
3,1.0
4,1.0


In [124]:
submission_mid_df = pd.read_csv('submission_mid.csv')
sample_submission_df = pd.read_csv('sample_submission.csv')

print("\n--- submission_mid.csv head ---")
display(submission_mid_df.head())

print("\n--- sample_submission.csv head ---")
display(sample_submission_df.head())


--- submission_mid.csv head ---


,id,STAR_WHITE_DWARF,STAR_CATY_VAR,STAR_BROWN_DWARF,SERENDIPITY_RED,REDDEN_STD,STAR_BHB,GALAXY,SERENDIPITY_DISTANT,QSO,SKY,STAR_RED_DWARF,ROSAT_D,STAR_PN,SERENDIPITY_FIRST,STAR_CARBON,SPECTROPHOTO_STD,STAR_SUB_DWARF,SERENDIPITY_MANUAL,SERENDIPITY_BLUE
0,199991,0.001783,0.006145,0.000013,0.006071,0.000256,0.000204,0.000139,0.002337,0.135623,0.000000,0.00000,0.000306,0.010733,0.000059,0.000257,0.000000,0.835922,0.000153,0.000000
1,199992,0.035121,0.430287,0.000002,0.174342,0.033099,0.014894,0.287350,0.000808,0.003287,0.001962,0.00000,0.004082,0.000422,0.006576,0.000104,0.000106,0.005805,0.000323,0.001430
2,199993,0.962330,0.009067,0.001846,0.006918,0.000088,0.000014,0.003803,0.000049,0.000009,0.000000,0.00017,0.000035,0.000015,0.010375,0.003399,0.000000,0.000654,0.000110,0.001116
3,199994,0.000372,0.170578,0.000000,0.032130,0.703540,0.039709,0.034347,0.000270,0.000012,0.000009,0.00000,0.004548,0.000000,0.000000,0.005341,0.000110,0.000018,0.000000,0.009018
4,199995,0.001403,0.003633,0.000013,0.006861,0.000010,0.000248,0.000339,0.000287,0.018411,0.000000,0.00000,0.000019,0.001402,0.000095,0.000562,0.000000,0.966527,0.000188,0.000000



--- sample_submission.csv head ---


,id,STAR_WHITE_DWARF,STAR_CATY_VAR,STAR_BROWN_DWARF,SERENDIPITY_RED,REDDEN_STD,STAR_BHB,GALAXY,SERENDIPITY_DISTANT,QSO,SKY,STAR_RED_DWARF,ROSAT_D,STAR_PN,SERENDIPITY_FIRST,STAR_CARBON,SPECTROPHOTO_STD,STAR_SUB_DWARF,SERENDIPITY_MANUAL,SERENDIPITY_BLUE
0,199991,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,199992,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,199993,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,199994,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,199995,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [126]:
import lightgbm as lgb

# LightGBM 모델 초기화 및 학습
model_lgbm = lgb.LGBMClassifier(
    objective='multiclass',
    num_class=len(le.classes_),
    n_estimators=100,
    learning_rate=0.05,
    num_leaves=31,
    max_depth=-1,
    random_state=42,
    n_jobs=-1
)

model_lgbm.fit(X, y)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.097735 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 6885
[LightGBM] [Info] Number of data points in the train set: 199991, number of used features: 27
[LightGBM] [Info] Start training from score -1.678020
[LightGBM] [Info] Start training from score -1.392670
[LightGBM] [Info] Start training from score -2.616019
[LightGBM] [Info] Start training from score -3.414238
[LightGBM] [Info] Start training from score -2.218199
[LightGBM] [Info] Start training from score -3.760545
[LightGBM] [Info] Start training from score -3.333681
[LightGBM] [Info] Start training from score -8.095154
[LightGBM] [Info] Start training from score -4.357484
[LightGBM] [Info] Start training from score -7.361841
[LightGBM] [Info] Start training from score -2.615198
[LightGBM] [Info] Start training from score -2.695583
[LightGBM] [Info] Start training from score -5.991420
[LightGB

LGBMClassifier(learning_rate=0.05, n_jobs=-1, num_class=19,
               objective='multiclass', random_state=42)

In [127]:
# 예측 확률 생성
proba_lgbm = model_lgbm.predict_proba(X_test)

# 제출 파일 생성
submission_lgbm = pd.read_csv('/content/sample_submission.csv')

# LabelEncoder에서 클래스 이름 가져오기
class_names = le.classes_

# 확률 DataFrame 생성 (컬럼 이름을 클래스 이름으로 설정)
proba_df_lgbm = pd.DataFrame(proba_lgbm, columns=class_names)

# submission_lgbm DataFrame에 확률 할당
submission_lgbm.iloc[:, 1:] = proba_df_lgbm

# CSV 파일로 저장
submission_lgbm.to_csv('submission_lgbm.csv', index=False)

print("LightGBM 모델로 예측된 submission_lgbm.csv 파일이 생성되었습니다.")
submission_lgbm.head()

LightGBM 모델로 예측된 submission_lgbm.csv 파일이 생성되었습니다.


,id,STAR_WHITE_DWARF,STAR_CATY_VAR,STAR_BROWN_DWARF,SERENDIPITY_RED,REDDEN_STD,STAR_BHB,GALAXY,SERENDIPITY_DISTANT,QSO,SKY,STAR_RED_DWARF,ROSAT_D,STAR_PN,SERENDIPITY_FIRST,STAR_CARBON,SPECTROPHOTO_STD,STAR_SUB_DWARF,SERENDIPITY_MANUAL,SERENDIPITY_BLUE
0,199991,0.002284,0.007259,0.000367,0.007592,0.000584,0.000362,0.000607,0.000000e+00,0.106130,6.672637e-07,0.000302,0.000325,6.135929e-04,0.000111,0.000202,1.061408e-87,0.873181,0.000027,0.000052
1,199992,0.010850,0.300882,0.000590,0.370056,0.001169,0.000493,0.313231,1.777864e-07,0.000115,8.886615e-41,0.000564,0.000608,1.097876e-05,0.000429,0.000332,6.925755e-212,0.000543,0.000027,0.000098
2,199993,0.996238,0.002215,0.000082,0.000872,0.000083,0.000026,0.000111,2.005825e-09,0.000009,1.479685e-07,0.000040,0.000060,7.119368e-07,0.000109,0.000032,0.000000e+00,0.000039,0.000003,0.000081
3,199994,0.000911,0.090642,0.000295,0.020525,0.860779,0.000432,0.024991,0.000000e+00,0.000058,6.368983e-07,0.000282,0.000310,4.990194e-06,0.000072,0.000283,0.000000e+00,0.000272,0.000019,0.000124
4,199995,0.000246,0.000789,0.000032,0.000777,0.000055,0.000036,0.000125,7.549886e-09,0.000064,1.251472e-51,0.000028,0.000030,6.553075e-06,0.000009,0.000019,4.488295e-147,0.997776,0.000002,0.000005


In [132]:
model_lgbm = lgb.LGBMClassifier(
    objective='multiclass',
    num_class=len(le.classes_),
    n_estimators=200,        # ↑ 늘림
    learning_rate=0.05,
    num_leaves=31,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,           # 추가
    reg_lambda=0.1,          # 추가
    random_state=42,
    n_jobs=-1
)